# MaintiDoc — extraction, vectorisation CLIP et ChromaDB

Adaptation du notebook de cours *RAG Multimodal Agentique* (Pr. Mohamed YOUSSFI) :
CLIP, extraction, aperçu, vectorisation, puis **indexation / recherche ChromaDB**.

Pas encore : `rag()` génératif multimodal ni agent LangChain.

## CLIP Model (smoke test)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from maintidoc.clip_smoke import encoder_exemple

info = encoder_exemple(
    texte="Panneau de commande d'une pompe",
    image=ROOT / "corpus" / "images" / "photo_panneau_pompe_a.jpg",
)
print("Device :", info["device"])
print("Dimension texte :", info["dimension_texte"])
print("Dimension image :", info["dimension_image"])
print("Embedding texte (5 premières valeurs) :", info["texte_5"])
print("Embedding image (5 premières valeurs) :", info["image_5"])

## Indiquer le document PDF du corpus

In [ ]:
from maintidoc.extraction import chemin_pdf, dossier_sortie

DOCUMENT_ID = "schema_pompe_a"
PDF_PATH = chemin_pdf(DOCUMENT_ID)
OUTPUT_DIR = dossier_sortie(DOCUMENT_ID)
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"

print(f"PDF : {PDF_PATH}")
print(f"Résultats : {OUTPUT_DIR}")

## Extraire le texte et les images

In [ ]:
from maintidoc.extraction import extraire_document

resultat = extraire_document(DOCUMENT_ID)
print(f"Pages traitées : {resultat.nombre_pages}")
print(f"Images enregistrées : {resultat.nombre_images}")
if resultat.pages_sans_texte:
    print("Pages sans texte détectable :", resultat.pages_sans_texte)

## Afficher un aperçu des résultats

In [ ]:
from IPython.display import display
from PIL import Image

from maintidoc.extraction import apercu_extraction

apercu = apercu_extraction(DOCUMENT_ID)
print("APERÇU DU TEXTE\n")
print(apercu["texte_apercu"])
print("\nAPERÇU DES IMAGES\n")
for chemin in apercu["images"]:
    fichier = Path(chemin)
    print(fichier.name)
    with Image.open(fichier) as image:
        vignette = image.copy()
        vignette.thumbnail((600, 600))
        display(vignette)

## Vectorisation CLIP (sans ChromaDB)

Découpage des textes en fragments, encodage texte/image avec CLIP, sauvegarde des vecteurs dans `data/processed/<document_id>/embeddings/embeddings.json`.

ChromaDB viendra ensuite.

In [ ]:
from maintidoc.vectorisation import apercu_vectorisation, vectoriser_document

resultat_vec = vectoriser_document(DOCUMENT_ID, extraire=False)
print(f"Fichier : {resultat_vec.fichier}")
print(f"Fragments texte : {resultat_vec.nombre_fragments_texte}")
print(f"Images : {resultat_vec.nombre_images}")
print(f"Dimension : {resultat_vec.dimension}")
print("\nAperçu :")
for ligne in apercu_vectorisation(DOCUMENT_ID, n=3):
    print(ligne)

## Indexation ChromaDB

Les embeddings déjà calculés (`embeddings.json`) sont upsertés dans une collection persistante `maintidoc_clip_vit_b32` sous `data/chroma/`.

In [ ]:
from maintidoc.chroma_index import (
    dossier_chroma,
    formater_resultats,
    indexer_document,
    ouvrir_collection,
    rechercher,
)

col = ouvrir_collection()
n = indexer_document(DOCUMENT_ID, collection=col)
print(f"ChromaDB : {dossier_chroma()}")
print(f"Éléments indexés pour {DOCUMENT_ID} : {n}")
print(f"Total collection : {col.count()}")

## Recherche multimodale

In [ ]:
resultats = rechercher(
    texte="température de palier pompe HX-200",
    type_resultat="texte",
    k=5,
    collection=col,
)
for ligne in formater_resultats(resultats):
    print(
        f"{ligne['rang']}. {ligne['type'].upper()} | "
        f"{ligne['document_id']} p.{ligne['page']} | "
        f"sim={ligne['similarite']:.3f}"
    )
    print(ligne["texte"][:200])
    print()